<a href="https://colab.research.google.com/github/Elight-dotcom/data-mining/blob/main/DM_M5_3124600100_Ardanu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M5 - Validation Model of Classification - Data Mining

Nama : Ardanu Egitya Ash Shafah  
NRP : 3124600100  
Kelas : D4 IT D

## Import Library

In [55]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, LeaveOneOut
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import MinMaxScaler

## Menampilkan dataset titanic.csv

In [56]:
data_path = '/content/drive/MyDrive/PENS/Semester 5/Data Mining/titanic.csv'
dataset = pd.read_csv(data_path)

dataset

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


## Ambil dataset tertentu dan isi missing value

In [57]:
features = ['Sex', 'Age', 'Pclass', 'Fare']

df_features = dataset[features].copy()

# Encoding Sex
df_features['Sex'] = df_features['Sex'].map({'male': 0, 'female': 1})

# Isi missing value dengan mean per class
df_features['Age'] = df_features.groupby('Pclass')['Age'].transform(lambda x: x.fillna(x.mean()))
df_features['Fare'] = df_features.groupby('Pclass')['Fare'].transform(lambda x: x.fillna(x.mean()))

df_features

,Sex,Age,Pclass,Fare
0,0,22.00000,3,7.2500
1,1,38.00000,1,71.2833
2,1,26.00000,3,7.9250
3,1,35.00000,1,53.1000
4,0,35.00000,3,8.0500
...,...,...,...,...
886,0,27.00000,2,13.0000
887,1,19.00000,1,30.0000
888,1,25.14062,3,23.4500
889,0,26.00000,1,30.0000


## Ambil dataset kolom survived

In [58]:
label = dataset['Survived']

# Mangubah data ke dalam bentuk array
y = label.values
X = df_features.values

label

,Survived
0,0
1,1
2,1
3,1
4,0
...,...
886,0
887,1
888,0
889,1


## Inisialisasi Algoritma Klasifikasi k-NN dan MinMax Scaler

In [59]:
knn = KNeighborsClassifier(n_neighbors=3)

## Validation Model

### Hold-out Method (70%-30%)

normalisasi

In [60]:
X_train_ho, X_test_ho, y_train_ho, y_test_ho = train_test_split(X, y, test_size=0.3, random_state=42)

ho_scaler = MinMaxScaler()

X_train_ho_scaled = ho_scaler.fit_transform(X_train_ho)
X_test_ho_scaled = ho_scaler.transform(X_test_ho)

min_ho = ho_scaler.data_min_
max_ho = ho_scaler.data_max_


print("--- Metode Hold-out (70-30) ---")
print(f"Nilai Min (Sex, Age, Pclass, Fare): {min_ho}")
print(f"Nilai Max (Sex, Age, Pclass, Fare): {max_ho}")

--- Metode Hold-out (70-30) ---
Nilai Min (Sex, Age, Pclass, Fare): [0.   0.42 1.   0.  ]
Nilai Max (Sex, Age, Pclass, Fare): [  1.      80.       3.     512.3292]


Menghitung error ratio

In [61]:


knn.fit(X_train_ho_scaled, y_train_ho)
err_ho = 1 - accuracy_score(y_test_ho, knn.predict(X_test_ho_scaled))

print(f"Error Ratio: {err_ho:.4f}\n")

Error Ratio: 0.1866



### K-Fold

Normalisasi dan menghitung error rasio

In [62]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)
err_kf = []
for train_idx, test_idx in kf.split(X):
    X_train_f, X_test_f = X[train_idx], X[test_idx]
    y_train_f, y_test_f = y[train_idx], y[test_idx]

    kfold_scaler = MinMaxScaler()

    X_train_s = kfold_scaler.fit_transform(X_train_f)

    X_test_s = kfold_scaler.transform(X_test_f)

    knn.fit(X_train_s, y_train_f)
    err_kf.append(1 - accuracy_score(y_test_f, knn.predict(X_test_s)))

print("--- Metode K-Fold (k=10) ---")
print(f"Rata-rata Error Ratio: {np.mean(err_kf):.4f}\n")

--- Metode K-Fold (k=10) ---
Rata-rata Error Ratio: 0.1751



### Leave-One-Out

In [63]:
loo = LeaveOneOut()
err_loo = []

for train_idx, test_idx in loo.split(X):
    X_train_l, X_test_l = X[train_idx], X[test_idx]
    y_train_l, y_test_l = y[train_idx], y[test_idx]

    scaler = MinMaxScaler()

    X_train_s = scaler.fit_transform(X_train_l)

    X_test_s = scaler.transform(X_test_l)

    knn.fit(X_train_s, y_train_l)
    err_loo.append(1 - accuracy_score(y_test_l, knn.predict(X_test_s)))

print("--- Metode LOO ---")
print(f"Rata-rata Error Ratio: {np.mean(err_loo):.4f}")

--- Metode LOO ---
Rata-rata Error Ratio: 0.1728
